# Imports

In [1]:
import os
import sys
import os.path as osp
import argparse

import numpy as np
import torch
import trimesh
import cv2

from scipy.spatial.distance import cdist
import plotly.graph_objects as go

In [2]:
sys.path.append("..")

In [3]:
from model.hand_opt import AdamGraspCmap
from utils.grasp_utils import get_handmodel, convert_9dGrasp_to_RT

from utils.pc_utils import (
    apply_extrinsics,
    estimate_normals_with_open3d,
    transform_to_camera_frame,
    compute_contact_map,
    compute_contact_map_aligned,
    get_fetch_gripper_mesh,
    get_gripper_pts_with_RT,
    determine_local_objpc_region,
    translate_grasp_along_palm_normal,
)

from utils.fig_utils import (
    plot_point_cloud,
    plot_point_cloud_cmap,
    plot_trimesh_mesh,
    viz_obj_pc_with_grasps,
)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
def get_q(RT, target_model):
    """
    Constructs a grasp q tensor (9-D) for an equivalent 4x4 RT pose
    """
    tra = RT[:3, 3]
    rot = RT[:3, :3]
    q = torch.zeros(9)
    q[:3] = torch.tensor(tra)
    q[3:] = torch.tensor(rot.T.reshape(-1)[:6])
    q = q.to(target_model.device)
    if q.shape[0] != 9 + len(target_model.dynamic_joints):
        # We optimized only for pose, so need to provide dummy joints
        q = torch.cat(
            (
                q,
                (
                    target_model.dynamic_joints_q_upper[0]
                    - target_model.dynamic_joints_q_mid[0]
                ),
            ),
            dim=0,
        )
    return q


# Input Data

In [5]:
input_fname = "../assets/data/dummy_opt_data.npz"
data = np.load(input_fname)
obj_pc = data["object_pc"]
RT_current = data["RT_grasp"]
RT_camera = data["RT_camera"]

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
target_gripper: str = "fetch_gripper"

# Hyper Params

In [7]:
THRESHOLD_DIST_LOCAL = 0.1 # object points farther than this (wrt gripper) will be clipped.
# NOTE: For partial point clouds, its best to use this as we don't have ground # truth normals to use the normal-aligned energy
ENERGY_FUNC = "euclidean_dist" 
SHARP_FACTOR = 10 # sharpening the contact goal -- more reliable target for opt
WT_COLLISION = 1
WT_CONTACT = 0.2
OPT_ONLY_TRANS = False
NUM_ITERS = 100 
NUM_PARALLEL_OPT = 4 # keep this number small to fit inside the gpu memory

# Setup: Gripper Model, Object Point Cloud

In [8]:
fetch_gripper_mesh = get_fetch_gripper_mesh()
target_model = get_handmodel(
        target_gripper,
        1,
        device,
        json_path="urdf_assets_meta.json",
        datadir="../grippers/",
)

In [9]:
############### Local Region of ObjPC ###############
# NOTE: Determine local region for the object pc using distance to fetch
# gripper mesh points
gripper_pts = get_gripper_pts_with_RT(
    fetch_gripper_mesh, RT_current
)
obj_pc_subset = determine_local_objpc_region(
    obj_pc, gripper_pts, dist_threshold=THRESHOLD_DIST_LOCAL, count=1000
)
############################################################

In [10]:
############### Object PC Normals & ContactMap ###############
objpcd_with_normals_in_world = estimate_normals_with_open3d(
    apply_extrinsics(obj_pc_subset, RT_camera), RT_camera[:3, 3]
)
# Get in camera frame
objpcd_with_normals_in_camera = transform_to_camera_frame(
    objpcd_with_normals_in_world, RT_camera
)
objpc_pts = np.array(obj_pc_subset)
objpc_nrm = np.asarray(objpcd_with_normals_in_camera.normals)
############################################################

# Cmap Goal for Optimization

In [11]:
############### Set Source Grasp + CMap Goal for Grasp Opt ###############

# NOTE: Any contact/rfp based grasp optimization needs a *contact goal* to
# optimize towards. That is usally provided by a learned model but in 1-shot
# unseen, partial object setting with an initial (noisy) grasp from a human demo
# we create a contact goal using the noisy grasp.

# NOTE: Here we create a "fake" standoff grasp by pushing the grasp towards 
# the object. This will give a strong contact map goal with the object.
# Setting the delta to be negative will set the gripper *away* from the object

# Create a standoff with 1cm (0.01m) *towards* the object
RT_standoff = translate_grasp_along_palm_normal(RT_current, delta=0.01)
q_standoff_grasp = get_q(RT_standoff, target_model)
source_q = q_standoff_grasp

# NOTE: Other Option: Just use the noisy grasp for the contact goal -- might
# not work if the object is far away --> bad contact goal to optimize towards.

# q_current_grasp = get_q(RT_current, target_model)
# source_q = q_current_grasp

# Use gripper surface points for the contact map goal
gripper_surf_pts_for_cmap = (
    target_model.get_surface_points(source_q.unsqueeze(0))[0].cpu().numpy()
)

if ENERGY_FUNC == "align_dist":
    contact_map = compute_contact_map_aligned(
        gripper_surf_pts_for_cmap, objpc_pts, objpc_nrm, SHARP_FACTOR
    )
else:
    contact_map = compute_contact_map(
        gripper_surf_pts_for_cmap, objpc_pts, SHARP_FACTOR
    )
############################################################

In [12]:
######### Grasp Opt Init: Contact Goal for Optimization #########

# # Construct the contact map goal:
# # Goal = [obj pc points (N,3), obj pc normals (N,3), contact map (N, 1)], Shape = (N, 7)

# Augmented Obj Pt Cloud
# NOTE: Initialize a thin shell around the object point cloud for collision
# consideration
# NOTE: 0.005 i.e 5mm used since using larger values effectively means that
# we are scaling up the object --> could create issues in the optimization
augmented_obj_pts = objpc_pts + 0.005 * objpc_nrm
cmap_goal = np.concatenate(
    [augmented_obj_pts, objpc_nrm, contact_map.reshape(-1, 1)], axis=1
)
cmap_tensor = torch.tensor(cmap_goal)
###########################################################################

# Run Grasp Opt

In [13]:
######### Run the Optimization #########
grasp_transfer_opt = AdamGraspCmap(
    target_robot_name=target_gripper,
    contact_weight=WT_CONTACT,
    collision_weight=WT_COLLISION,
    opt_only_trans=OPT_ONLY_TRANS,
    sharp_factor=SHARP_FACTOR,
    source_grasp=source_q,
    num_particles=NUM_PARALLEL_OPT,
    learning_rate=1e-3,
    max_iter=NUM_ITERS,
    device=device,
    energy_func_name=ENERGY_FUNC,
)

q_traj, energy_traj, econ_traj, epen_traj, _ = grasp_transfer_opt.run_adam(
    contact_map_goal=cmap_tensor, source_grasp=source_q, running_name="test"
)

energy_traj = np.asarray(energy_traj).T  # shape (num_parallel, num_iters)
econ_traj = np.asarray(econ_traj).T  # shape (num_parallel, num_iters)
epen_traj = np.asarray(epen_traj).T

min_energy_index = energy_traj[:, -1].argmin()
best_q = q_traj[min_energy_index, -1]

if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
    # We optimized only for pose, so need to provide dummy joints
    best_q = torch.cat(
        (
            best_q,
            (
                target_model.dynamic_joints_q_upper[0]
                - target_model.dynamic_joints_q_mid[0]
            ),
        ),
        dim=0,
    )
RT_optimized_grasp = convert_9dGrasp_to_RT(best_q)
########################################

opt_iters: 100%|██████████| 100/100 [00:03<00:00, 31.22it/s]


# Visualize Results

In [14]:
############### GrasOpt Result Viz ################
# VIZ: Local Obj PC region and Gripper Pts + Contact Map
vis_data = []
vis_data += [plot_point_cloud_cmap(objpc_pts, color_levels=contact_map, size=4)]

vis_data += [
    plot_trimesh_mesh(
        fetch_gripper_mesh.copy().apply_transform(RT_current),
        color="red",
        opacity=0.4,
    )
]

vis_data += [
    plot_trimesh_mesh(
        fetch_gripper_mesh.copy().apply_transform(RT_optimized_grasp),
        color="lightgreen",
        opacity=0.4,
    )
]
fig = go.Figure(data=vis_data)
fig.show()
############################################################